# Agentic Artificial Intelligence
## Exercise - Unit 04: Working with prompts

Welcome to the fourth unit of the Agentic Artificial Intelligence course!

## Learning Objectives
By the end of this lesson, students will:
1. Understand the purpose and structure of prompt templates
2. Learn how to use prompt templates with placeholders
3. Understand how to format prompts with dynamic values (name, date, etc.)
4. Integrate prompts from `prompts.py` into agent classes
5. Follow LangGraph best practices for system messages

## Prerequisites
- Students should have completed Unit 03 exercises on LangGraph basics
- Understanding of Python string formatting (f-strings, `.format()`, etc.)
- Familiarity with the `BaseAgent`, `SimpleAgent`, and `ToolAgent` classes

## Step 1: Starting with Simple LLM Calls

Let's begin with the most basic approach - calling an LLM directly without any prompts. This helps us understand the foundation before adding complexity.

### 1.1: Direct LLM Call Without Prompts

First, let's see what happens when we call an LLM with just a user message - no system prompt, no instructions:


In [ ]:
# Import necessary libraries
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

# Initialize an LLM
llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

# Call the LLM with just a user message (no system prompt)
user_message = HumanMessage(content="What is 2 + 2?")
response = llm.invoke([user_message])
response.pretty_print()

Notice:
- The LLM responds, but we have no control over its behavior
- We can't tell it how to act, what role to play, or what style to use
- This is why we need prompts!

## Step 2: Adding Basic Prompts

Now let's add a simple, hardcoded prompt to guide the LLM's behavior. This is the most straightforward way to use prompts.


In [ ]:
# Import SystemMessage for system prompts
from langchain_core.messages import SystemMessage, HumanMessage

# Create a simple hardcoded prompt
simple_prompt = "You are a helpful math tutor. Always explain your reasoning step by step."

# Call LLM with system message + user message
messages = [
    SystemMessage(content=simple_prompt),
    HumanMessage(content="What is 2 + 2?")
]

response = llm.invoke(messages)
response.pretty_print()

Notice:
- The LLM now follows the instructions in the system prompt
- It acts as a math tutor and explains step by step
- But the prompt is hardcoded - we can't easily change it or reuse it

### 2.1: Comparing Different Prompts

Let's see how different prompts change the LLM's behavior:


Key Insight:
- The same question gets different responses based on the prompt
- Prompts are powerful tools for controlling LLM behavior
- But hardcoding prompts makes it hard to reuse and maintain them

## Step 3: Introducing Prompt Templates

Instead of hardcoding prompts, we can create **prompt templates** - reusable strings that we can format with different values. This makes our prompts more flexible and maintainable.

In [ ]:
# Create a simple prompt template (a string with placeholders)
# Placeholders use {variable_name} syntax
greeting_template = "Hello! My name is {name}. I am a {role}."

# We can format this template with different values
greeting1 = greeting_template.format(name="Alice", role="teacher")
greeting2 = greeting_template.format(name="Bob", role="engineer")

print("Template:", greeting_template)
print("\nFormatted version 1:", greeting1)
print("Formatted version 2:", greeting2)

print("\n" + "="*60)
print("Using template with LLM:")
print("="*60)

# Create a prompt template for our LLM
prompt_template = "You are a helpful {role} assistant. Always be {tone}."

# Format it with specific values
formatted_prompt = prompt_template.format(role="math tutor", tone="patient and encouraging")

messages = [
    SystemMessage(content=formatted_prompt),
    HumanMessage(content="Can you explain fractions?")
]

response = llm.invoke(messages)
response.pretty_print()


Benefits of templates:
- Reusable: Same template, different values
- Maintainable: Change template once, affects all uses
- Flexible: Easy to customize for different scenarios

### 3.1: Understanding Placeholders

Placeholders in prompt templates use the `{variable_name}` syntax. When you format a template, you provide values for these placeholders using Python's `.format()` method.

In [ ]:
# Example: Template with multiple placeholders
template = "Hello, I am {name}. Today is {date}. I work as a {job}."

# Format with all required values
formatted = template.format(
    name="Alice",
    date="2024-01-15",
    job="data scientist"
)

print("Template:", template)
print("Formatted:", formatted)

print("\n" + "="*60)
print("Common Pitfall: Missing Placeholders")
print("="*60)

# What happens if we forget a placeholder?
try:
    bad_result = template.format(name="Bob", date="2024-01-15")
    # Missing 'job' placeholder!
except KeyError as e:
    print(f"Error: Missing placeholder '{e.args[0]}'")
    print("\nAlways provide ALL required placeholders!")

print("\n" + "="*60)
print("Tip: Check placeholders before formatting")
print("="*60)

import re
placeholders = re.findall(r'\{(\w+)\}', template)
print(f"Template requires these placeholders: {', '.join(placeholders)}")
print("Make sure to provide values for all of them!")


## Step 4: Prompt Templates with Dynamic Values

Now let's create more sophisticated templates that include dynamic values like dates and other context that changes over time.

### 4.1: The Problem with Static Date Formatting

When we format a prompt template with a date using `.format()`, the date is captured **once** at the time of formatting. This means if you create an agent and use it multiple times, the date will always be the same - it won't update!

Let's see the problem:


In [ ]:
# Example: The problem with static date formatting
from datetime import datetime
import time

# Create a template with a date placeholder
template = "You are a helpful assistant. Today's date is {date}."

# Format it once - date is captured at this moment
formatted_prompt = template.format(date=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Formatted prompt (first time):")
print(formatted_prompt)

# Wait a few seconds
print("\nWaiting 3 seconds...")
time.sleep(3)

# Format it again - but the date is STILL the same!
formatted_prompt2 = template.format(date=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("\nFormatted prompt (after 3 seconds):")
print(formatted_prompt2)

print("\n" + "="*60)
print("Problem: Each time we format, we need to manually call datetime.now()")
print("If we create an agent with a formatted prompt, the date won't update!")
print("="*60)

### 4.2: Solution - LangChain's Partial Prompt Templates

LangChain provides a solution using **partial prompt templates**. The key idea is to use `.partial()` to pre-fill variables with **functions** (not values). When you format the prompt later, the function is called each time, ensuring dynamic values are always up-to-date.

**Key Concept:** Pass a **function** to `.partial()`, not a value!


In [ ]:
# Step 1: Import LangChain's PromptTemplate
from langchain.prompts import PromptTemplate

# Step 2: Create a function that returns the current time
def get_current_time():
    """Function that returns the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Step 3: Create a PromptTemplate with the placeholder
template = "You are a helpful assistant. Today's date is {date}."
prompt = PromptTemplate(
    input_variables=["date"],  # List all variables that need to be filled
    template=template
)

# Step 4: Use .partial() to pre-fill with a FUNCTION (not a value!)
# This is the key: pass the function itself, not the result of calling it
partial_prompt = prompt.partial(date=get_current_time)

print("Step-by-step demonstration:")
print("="*60)
print("\n1. Created PromptTemplate with placeholder: {date}")
print("2. Created function get_current_time() that returns current time")
print("3. Used .partial(date=get_current_time) - passing the FUNCTION")
print("\n" + "="*60)

In [ ]:
# Step 5: Format the prompt - get_current_time() is called NOW
formatted1 = partial_prompt.format()
print("First format (time 1):")
print(formatted1)

# Wait a few seconds
print("\nWaiting 3 seconds...")
time.sleep(3)

# Step 6: Format again - get_current_time() is called AGAIN (new time!)
formatted2 = partial_prompt.format()
print("\nSecond format (time 2, after 3 seconds):")
print(formatted2)

print("\n" + "="*60)
print("✅ Success! The date/time is now dynamically updated each time!")
print("="*60)

### 4.3: Critical Difference - Function vs. Value

**❌ WRONG - Passing a value (captured once):**
```python
# This captures the time ONCE when .partial() is called
partial_prompt = prompt.partial(date=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
```

**✅ CORRECT - Passing a function (called each time):**
```python
# This calls get_current_time() EACH TIME .format() is called
partial_prompt = prompt.partial(date=get_current_time)
```

Let's see the difference:


In [ ]:
# WRONG WAY: Passing a value (static)
print("❌ WRONG WAY - Passing a value:")
print("="*60)
wrong_prompt = PromptTemplate(
    input_variables=[],
    template="Current time: {time}"
)
# This captures the time ONCE
wrong_partial = wrong_prompt.partial(time=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Time captured:", wrong_partial.format())
time.sleep(2)
print("After 2 seconds:", wrong_partial.format())  # Same time!

print("\n" + "="*60)
print("✅ CORRECT WAY - Passing a function:")
print("="*60)

# CORRECT WAY: Passing a function (dynamic)
def get_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

correct_prompt = PromptTemplate(
    input_variables=[],
    template="Current time: {time}"
)
# This calls get_time() EACH TIME
correct_partial = correct_prompt.partial(time=get_time)
print("Time 1:", correct_partial.format())
time.sleep(2)
print("Time 2 (after 2 seconds):", correct_partial.format())  # New time!

### 4.4: Practical Example - Using Partial Prompts with Agents

Now let's see how to use this in practice with an agent. We'll create a prompt template that always has the current date:


In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

# Create a function for getting current date
def get_current_date():
    return datetime.now().strftime("%Y-%m-%d")

# Create prompt template
agent_template = """You are a helpful assistant called {name}. 
Today's date is {date}.

Always mention the current date when relevant to the conversation."""

prompt = PromptTemplate(
    input_variables=["name", "date"],
    template=agent_template
)

# Use partial to pre-fill date with function, but keep name as variable
partial_prompt = prompt.partial(date=get_current_date)

# Now we only need to provide 'name' when formatting
formatted_prompt = partial_prompt.format(name="ResearchBot")
print("Formatted prompt with dynamic date:")
print("="*60)
print(formatted_prompt)

# Use it with LLM
messages = [
    SystemMessage(content=formatted_prompt),
    HumanMessage(content="What's today's date?")
]

response = llm.invoke(messages)
print("\n" + "="*60)
print("LLM Response:")
print("="*60)
response.pretty_print()

### 4.5: Key Takeaways

**Summary of Dynamic Values in Prompt Templates:**

1. **Problem**: Using `.format()` with `datetime.now()` captures the time once, not dynamically
2. **Solution**: Use LangChain's `PromptTemplate` with `.partial()` method
3. **Critical Point**: Pass a **function** to `.partial()`, not a value
4. **Result**: The function is called each time `.format()` is invoked, ensuring fresh values

**Pattern to Remember:**
```python
from langchain.prompts import PromptTemplate
from datetime import datetime

def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

prompt = PromptTemplate(
    input_variables=["time"],
    template="Current time: {time}"
)

# Pass the FUNCTION, not the value!
partial_prompt = prompt.partial(time=get_current_time)

# Each call to format() gets a fresh time
formatted = partial_prompt.format()
```

**When to Use:**
- Dates and times that should update on each request
- Any value that needs to be computed fresh each time
- Context that changes between requests (user location, system status, etc.)


In [ ]:
# Test with different prompts
prompts_to_test = [
    ("Friendly assistant", "You are a friendly and cheerful assistant."),
    ("Professional consultant", "You are a professional business consultant. Be concise and formal."),
    ("Creative writer", "You are a creative writer. Use vivid language and storytelling.")
]

user_question = "Tell me about artificial intelligence."

for role, prompt_text in prompts_to_test:
    print(f"\n{'='*60}")
    print(f"Role: {role}")
    print(f"Prompt: {prompt_text}")
    print("="*60)
    
    messages = [
        SystemMessage(content=prompt_text),
        HumanMessage(content=user_question)
    ]
    
    response = llm.invoke(messages)
    print(f"\nResponse:")
    print(response.content[:200] + "...\n")

## Step 5: Using Prompts in Agent Classes

Now let's see how to use formatted prompts in our agent classes. We'll create a `DynamicPromptAgent` using the `system_prompt` from `prompts.py`:

In [ ]:
from langchain.chat_models import init_chat_model
from agentic_ai.agents.dynamic_prompt_agent import DynamicPromptAgent
from langgraph.checkpoint.memory import InMemorySaver
from agentic_ai.prompts.prompts import system_prompt

llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
memory = InMemorySaver()

dynamic_agent = DynamicPromptAgent(llm=llm, name="DynamicAgent", system_prompt=system_prompt, checkpointer=memory)

dynamic_agent.run("Hello! What is your name and what time is it?", thread_id="test_thread")

In [ ]:
dynamic_agent.run("Hello! What is your name and what time is it?", thread_id="test_thread")

In [ ]:
dynamic_agent.name = "OtherNameAgent"
dynamic_agent.run("Hello! What is your name and what time is it?", thread_id="test_thread")

## Step 6: Introducing Prompting techniques

In the lecture, we learned about some prompting techniques such as chain of thought and tree of thoughts. This is a good point to experiment with them to understand them better.

### 6.1: Chain of Thoughts (CoT) prompts
Chain of Thoughts (CoT) is a prompting technique that encourages step-by-step reasoning.

Instead of jumping directly to an answer, you should:
1. Break down the problem into smaller steps
2. Show your reasoning process for each step
3. Build upon previous steps to reach a conclusion
4. Clearly state intermediate conclusions before the final answer

This approach helps with complex problems that require logical reasoning, mathematical calculations, or multi-step analysis.
"""

In [ ]:
from agentic_ai.prompts.prompts import chain_of_thoughts_test_prompt

print(chain_of_thoughts_test_prompt)

In [ ]:
apples_task = 'Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does Roger have now? A: The answer is 11. Q: The cafeteria has 46 apples. If they use 16 to make lunch and bought 2 more packages with 6 apples each, how many apples does the cafeteria have left? A: The answer is '
response_standard_prompt = llm.invoke(apples_task)
response_standard_prompt.pretty_print()

In [ ]:
chain_of_thoughts_prompt_apples = chain_of_thoughts_test_prompt.format(problem=apples_task)
response_chain_of_thoughts_prompt = llm.invoke(chain_of_thoughts_prompt_apples)
response_chain_of_thoughts_prompt.pretty_print()

### 6.2: Tree of Thoughts (ToT) prompts
tree_of_thoughts_explanation = """Tree of Thoughts (ToT) is an advanced reasoning technique that explores multiple reasoning paths simultaneously.

Instead of following a single chain of reasoning, you should:
1. Generate multiple possible approaches or solutions
2. Evaluate each approach independently
3. Compare the strengths and weaknesses of each path
4. Select the best path or combine insights from multiple paths
5. Continue exploring branches from promising paths

This approach is useful for problems with multiple valid solutions, creative tasks, or when you need to explore different strategies before committing to one.
"""

In [ ]:
from agentic_ai.prompts.prompts import tree_of_thoughts_test_prompt

print(tree_of_thoughts_test_prompt)

In [ ]:
tree_of_thoughts_prompt_apples = tree_of_thoughts_test_prompt.format(problem=apples_task)
response_tree_of_thoughts_prompt = llm.invoke(tree_of_thoughts_prompt_apples)
response_tree_of_thoughts_prompt.pretty_print()

## LangGraph Best Practices for System Messages

According to LangGraph best practices, system messages should be:

1. **Added at the beginning** of conversations
2. **Added only once per thread** - Check for existing SystemMessage before adding
3. **Preserved in the state** - The system message is preserved in the state
4. **Use SystemMessage class** - Proper message type for system instructions

The `BaseAgent` class already implements these best practices in the `run()` method. Let's verify this:


In [ ]:
# Check conversation history to see system message handling
history = dynamic_agent.get_conversation_history(thread_id="test_thread")
print("Conversation history:")
print("=" * 50)
for i, msg in enumerate(history):
    print(f"{i+1}. {msg.__class__.__name__}: {msg.content[:100]}...")
    
print("\nNotice:")
print("- SystemMessage is added at the beginning")
print("- It's preserved across multiple messages")
print("- It's only added once per thread")

# Test with another message in the same thread
print("\n" + "="*60)
print("Sending another message in the same thread:")
print("="*60)
response2 = dynamic_agent.run("Can you help me research Python?", thread_id="test_thread")
response2['messages'][-1].pretty_print()

# Check history again
history2 = dynamic_agent.get_conversation_history(thread_id="test_thread")
print(f"\nTotal messages in thread: {len(history2)}")
print("System message is still there and preserved!")

## Creating Custom Prompt Templates

You can also create your own prompt templates! Here's an example for a specialized agent:

## IMPORTANT note:

If you want to include other variables that are dynamically loaded you have to implement respective functions and add them in the `init_prompt` function as done in the `DynamicPromptAgent` class. The logic below only works as we already implemented functions to dynamically load `{name}` and `{date}`

In [ ]:
# Example: Custom prompt template for a math tutor agent
math_tutor_prompt_template = """You are a helpful math tutor.

This is the latest information that you retrieve in real-time that you should use to answer the user's request:
- Your name is: {name}.
- The current date and time is: {date}.

Your role is to:
- Explain mathematical concepts clearly
- Provide step-by-step solutions
- Help students understand problem-solving strategies
- Encourage students to think through problems

Always be patient and encouraging. If a student makes a mistake, guide them to find the correct answer rather than just giving it.
"""

# Create agent with custom prompt
math_agent = DynamicPromptAgent(
    llm=llm,
    name="Elias",
    system_prompt=math_tutor_prompt_template,
    checkpointer=memory
)

# Test the math tutor agent
print("Testing custom math tutor prompt:")
print("=" * 50)
response = math_agent.run("Can you help me solve 2x + 5 = 15?", thread_id="math_thread")
response['messages'][-1].pretty_print()

In [ ]:
math_agent.run("What's your name and what time is it?", thread_id="math_thread")

## Summary: Working with Prompts

**Key Takeaways:**

1. **Start simple** - Begin with basic LLM calls, then add prompts
2. **Hardcoded prompts** - Simple strings work for one-off cases
3. **Prompt templates** - Use `{placeholder}` syntax for reusable prompts
4. **Dynamic values** - Format templates with `.format()` method
5. **Prompts subpackage** - Use pre-built templates from `agentic_ai.prompts.prompts`
6. **Agent integration** - Pass formatted prompts to agents via `system_prompt` parameter
7. **LangGraph best practices** - System messages are handled automatically by `BaseAgent`

**Common Pattern:**
```python
from agentic_ai.prompts.prompts import system_prompt
from datetime import datetime

formatted = system_prompt.format(
    name="AgentName",
    date=datetime.now().strftime("%Y-%m-%d")
)

agent = SimpleAgent(llm, name="AgentName", system_prompt=formatted)
```

**Progression We Learned:**
1. Simple LLM calls (no prompts)
2. Basic hardcoded prompts
3. Prompt templates (static)
4. Templates with dynamic values (dates, names)
5. Using the prompts subpackage

Now you're ready to use prompts in your own agent implementations!

# Exercise

In this exercise, you will build a specialized agent that demonstrates your understanding of LangGraph, state management, prompt templates, and object-oriented programming.

## Task 1: Create a Custom Agent Class with Prompt Templates

Create a new agent class called `RolePlayAgent` that extends `BaseAgent`. This agent should:

1. **Take a role/personality as a parameter** (e.g., "wizard", "detective", "chef", "scientist")
2. **Use a prompt template** from `prompts.py` or create a custom one that describes the agent's role and personality
3. **Format the prompt** with the agent's name and current date
4. **Implement the graph building** with at least one custom node (you can start with the basic llm_node pattern)
5. **Use memory** with `InMemorySaver` for conversation persistence

**Requirements:**
- The agent should introduce itself according to its role when first meeting a user
- The agent should maintain character throughout the conversation
- The agent should remember previous conversation context within the same thread
- **Use prompt templates** - either from `prompts.py` or create your own custom template

**Hints:**
- Review how `DynamicPromptAgent` extends `BaseAgent`
- Import and format a prompt template (or create your own) before passing it to `system_prompt`
- Use `datetime.now().strftime("%Y-%m-%d")` for date formatting
- Remember to use `thread_id` in the `run()` method to maintain conversation history
- You can create the agent in a new cell, or create it in a Python file and import it

**Example prompt template structure:**
```python
role_prompt_template = """You are a {role} called {name}.

This is the latest information that you retrieve in real-time that you should use to answer the user's request:
- The current date and time is: {date}.
- Your current mood is: {mood}.

Your personality and behavior:
- [Describe role-specific traits]
- [Add role-specific instructions]

</Instructions>
- When interacting with others always consider you personality, behaviour and you current mood.
- Remember to stay in character throughout the conversation.
</Instructions>
"""
```

! With the class SimpleAgent the exerise should work -> work with the class base 
Try the chat method!! Quite Useful, it is an infinite while loop, add a codeline 

BaseAgent class: Create a new agent: Task: RolePlayAgendt (BaseAgent)

Its the basic structure for every agent 

Especially understand the def run method 

To keep only the current message : modify the state 




### Task 1.1: Create Your RolePlayAgent

Start by creating your `RolePlayAgent` class. You can use the starter code below or create your own:


In [ ]:
# TODO: Create your RolePlayAgent class here
# You can use the starter code below or write your own implementation

from agentic_ai.agents.base import BaseAgent, State
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from datetime import datetime

# Example role prompt template
role_prompt_template = """You are a {role} called {name}. Today's date is {date}.

Your personality and behavior:
- You embody the characteristics of a {role}
- Stay in character throughout all conversations
- Be creative and engaging while maintaining your role

Remember to introduce yourself according to your role when meeting someone new.
"""

class RolePlayAgent(BaseAgent):
    def __init__(self, llm, role="assistant", checkpointer=None):
        # TODO: Format the prompt template with role, name, and date
        # TODO: Call super().__init__() with the formatted prompt
        pass
    
    def _build_graph(self):
        # TODO: Build the graph with nodes
        # Hint: Similar to SimpleAgent, but you can customize it
        pass
    
    def llm_node(self, state: State):
        # TODO: Implement LLM node logic
        # Hint: Use self.add_system_message() to ensure system message is added
        pass


## Task 2: Test Your Agent

Test your `RolePlayAgent` with the following scenarios:

1. **Initial greeting**: Send a message introducing yourself (e.g., "Hi, I'm Alice")
2. **Role consistency**: Ask the agent about their role/personality (e.g., "What are you?")
3. **Memory test**: In a later message, ask the agent to recall your name (e.g., "What's my name?")
4. **Thread isolation**: Create a second conversation with a different `thread_id` and verify it doesn't remember the first conversation
5. **Prompt verification**: Check that the system prompt was formatted correctly with name and date

**Expected behavior:**
- The agent should respond in character according to its role
- The agent should remember your name from the first conversation
- Different threads should have separate conversation histories
- The system prompt should include the agent's name and current date


In [ ]:
# TODO: Test your RolePlayAgent here


## Task 3 (Bonus): Agent2Agent Roleplay

Create a roleplay scenario where two `RolePlayAgent` instances interact with each other in a conversation. This exercise will help you understand how agents can communicate and maintain separate conversation contexts.

### Requirements:

1. **Create two agents with different roles**: Initialize two `RolePlayAgent` instances with distinct roles/personalities (e.g., "wizard" and "knight", "detective" and "suspect", "chef" and "food critic", etc.)

2. **Set up separate conversation threads**: Each agent should have its own `thread_id` to maintain separate conversation histories

3. **Implement a conversation loop**: Create a function or loop that:
   - Starts with an initial message from one agent (or a neutral prompt)
   - Takes agent A's response and sends it as a message to agent B
   - Takes agent B's response and sends it as a message to agent A
   - Continues for a specified number of turns (e.g., 3-5 turns)

4. **Display the conversation**: Print or display the conversation in a readable format, showing which agent said what

### Hints:

- Each agent needs its own `InMemorySaver` checkpointer (or use different `thread_id` values)
- Use different `thread_id` values for each agent (e.g., "agent1_thread" and "agent2_thread")
- Extract the text content from each agent's response using `response['messages'][-1].content`
- You can add context to each message, such as "Agent A says: [message]" to help the receiving agent understand who they're talking to
- Consider adding a system message or context about the other agent's role to make the conversation more engaging

### Example Structure:

```python
# Initialize two agents with different roles
agent1 = RolePlayAgent(llm=llm, role="wizard", checkpointer=InMemorySaver())
agent2 = RolePlayAgent(llm=llm, role="knight", checkpointer=InMemorySaver())

# Start the conversation
initial_message = "Hello! I'm looking for someone to help me on a quest."

# Conversation loop
for turn in range(5):
    # Agent 1 responds
    response1 = agent1.run(initial_message, thread_id="agent1_thread")
    agent1_message = response1['messages'][-1].content
    print(f"Wizard: {agent1_message}\n")
    
    # Agent 2 responds to agent 1
    response2 = agent2.run(agent1_message, thread_id="agent2_thread")
    agent2_message = response2['messages'][-1].content
    print(f"Knight: {agent2_message}\n")
    
    # Update initial_message for next turn
    initial_message = agent2_message
```

### Expected Behavior:

- Each agent should stay in character according to their role
- The conversation should flow naturally, with each agent responding to the other
- Agents should maintain their own conversation history within their respective threads
- The interaction should demonstrate how agents can communicate and build upon each other's responses

### Bonus Challenges:

- Add a moderator agent that observes and occasionally interjects
- Implement a turn-taking system where agents can "pass" if they have nothing to say
- Create a scenario where agents have conflicting goals and must negotiate
- Add memory sharing between agents (advanced)
